In [5]:
import json
import anthropic
import os
from tqdm import tqdm

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

In [21]:
with open('../data/val_data_2025_onward.jsonl', 'r') as f:
    original_posts = [json.loads(line) for line in f]

with open('outputs/checkpoint-800-generation-all-ok.jsonl', 'r') as f:
    ckpt_800_posts = [json.loads(line) for line in f]

with open('outputs/checkpoint-1628-generation-all-ok.jsonl', 'r') as f:
    ckpt_1628_posts = [json.loads(line) for line in f]

In [16]:
def parse_article(text):
    content_marker = '\n\n### Content: \n'
    title_marker = '### Title: '
    end_title_idx = text.find(content_marker)
    title = text[len(title_marker) : end_title_idx]
    content = text[end_title_idx + len(content_marker) : ]
    content = content.replace('<|end_of_text|>', '')
    content = content.replace(' ,', ',')
    content = content.replace(' .', '.')

    return title, content

In [22]:
posts = {}
for post in original_posts:
    title, content = parse_article(post['text'])
    posts[title] = {'original': content}

for post in ckpt_800_posts:
    title, content = parse_article(post['text'])
    posts[title]['ckpt_800'] = content


for post in ckpt_1628_posts:
    title, content = parse_article(post['text'])
    posts[title]['ckpt_1628'] = content

Choosing between the 2 generated posts:

In [40]:
for title, contents in tqdm(posts.items()):
    original_post = contents['original']
    ckpt_800_post = contents['ckpt_800']
    ckpt_1628_post = contents['ckpt_1628']

    prompt = (
        "You are a content quality evaluator. You will be given 3 blog posts, one from the original author, one from author coded 800, and one from author coded 1628. "
        "Based on the blog contents, your task is to pick one of the 2 posts from the coded authors, which one sounds better and have meaning more closely related to the original post.\n"
        "Output only 800 or 1628. Nothing else.\n\n"
        f"Title: {title}\n\n"
        f"Original Post:\n```\n{original_post}\n```\n\n"
        f"Post from author 800:\n```\n{ckpt_800_post}\n```\n\n"
        f"Post from author 1628:\n```\n{ckpt_1628_post}\n```\n\n"
        "Again, only respond with 800 or 1628 to indicate your selection of the better post. Nothing else."
    )
    while True:
        try:
            response = client.messages.create(
                model="claude-3-opus-20240229",
                max_tokens=5,
                messages=[{"role": "user", "content": prompt}]
            )
            decision = response.content[0].text.strip().lower()
            if decision not in ['800', '1628']:
                raise Exception(f'Wrong output: {decision}')
        except Exception as e:
            print(f"[ERROR] Claude evaluation failed: {e}")
            continue

        posts[title]['selection'] = decision
        break


100%|██████████| 102/102 [06:22<00:00,  3.75s/it]


In [ ]:
for title, contents in tqdm(posts.items()):
    ckpt_800_post = contents['ckpt_800']
    ckpt_1628_post = contents['ckpt_1628']

    prompt = (
        "You are a content quality evaluator. You will be given 2 blog posts, one from author coded 800, and one from author coded 1628. "
        "Given these 2 posts, which one sounds better in your opinion? If you only choose 1 to publish, which one would you choose?"
        "Output only 800 or 1628. Nothing else.\n\n"
        f"Title: {title}\n\n"
        f"Original Post:\n```\n{original_post}\n```\n\n"
        f"Post from author 800:\n```\n{ckpt_800_post}\n```\n\n"
        f"Post from author 1628:\n```\n{ckpt_1628_post}\n```\n\n"
        "Again, only respond with 800 or 1628 to indicate your selection of the better post. Nothing else."
    )
    while True:
        try:
            response = client.messages.create(
                model="claude-3-opus-20240229",
                max_tokens=5,
                messages=[{"role": "user", "content": prompt}]
            )
            decision = response.content[0].text.strip().lower()
            if decision not in ['800', '1628']:
                raise Exception(f'Wrong output: {decision}')
        except Exception as e:
            print(f"[ERROR] Claude evaluation failed: {e}")
            continue

        posts[title]['selection_2'] = decision
        break


In [58]:
from collections import Counter

res = []
for content in posts.values():
    res.append(content['selection'])

print(Counter(res))

Counter({'1628': 72, '800': 30})


The overfitted checkpoint (1628) generated more favorable posts than the one with lowest eval loss (800).

In [60]:
with open('./outputs/all_posts_with_claude_selection.json', 'w') as f:
    json.dump(posts, f)

In [11]:
with open('./outputs/all_posts_with_claude_selection.json', 'r') as f:
    posts = json.load(f)

In [22]:
final_result = {}
for title, content in posts.items():
    final_result[title] = {
        "original": content["original"],
        "generated": content['ckpt_800'] if content['selection'] == '800' else content['ckpt_1628']
    }


In [ ]:
"""
Generate an HTML comparison file with shuffled columns and hidden labels.
Labels now appear in the bottom-right corner of each article.
"""

from pathlib import Path
import html, random, re


def add_blank_lines(text: str) -> str:
    """Insert an empty line between every paragraph."""
    return re.sub(r"\n(?!\n)", "\n\n", text.strip())


def make_html(results: dict, out_path: str | Path = "comparison.html") -> None:
    out_path = Path(out_path)

    head = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Original vs AI-Generated Blog Posts</title>
<style>
  body          { font-family: Arial, sans-serif; margin: 0; padding: 24px; line-height: 1.6;
                   background: #f8f9fa; }
  h1            { margin-top: 0; }
  .post         { margin-bottom: 64px; }
  .comparison   { display: grid; grid-template-columns: 1fr 1fr; gap: 24px; }
  .article      { position: relative; background: #fff; border: 1px solid #ddd;
                  border-radius: 6px; padding: 20px; overflow: hidden; }
  .article pre  { margin: 0; white-space: pre-wrap; }
  /* ⬇️  CHANGED: move label to bottom-right */
  .label        { position: absolute; bottom: 12px; right: 12px; font-size: 12px;
                  padding: 4px 8px; background: rgba(255,255,255,0.9); border: 1px solid #777;
                  border-radius: 4px; }
  .hidden       { display: none; }
  .toggle-btn   { margin-top: 16px; padding: 6px 10px; font-size: 14px;
                  border: 1px solid #0077cc; color: #0077cc; background: #fff;
                  border-radius: 4px; cursor: pointer; }
  .toggle-btn:hover { background: #eef7ff; }
</style>
<script>
function reveal(btn) {
  const labels = btn.parentElement.querySelectorAll('.label');
  labels.forEach(l => l.classList.toggle('hidden'));
  btn.textContent = labels[0].classList.contains('hidden') ? 'Show source' : 'Hide source';
}
</script>
</head>
<body>
<h1>Original vs AI-Generated Blog Posts</h1>

<div style="background: #fff; border: 1px solid #ddd; border-radius: 6px; padding: 20px; margin-bottom: 32px;">
  <h2>Project Overview</h2>
  <p>This project fine-tuned <strong>Llama 3.1 70B</strong> on blog posts from <a href="https://nik.art/" target="_blank">nik.art</a> to generate content in the author's style. The model was trained on all posts published through 2024 and tested on posts from 2025 onward.</p>
  
  <h3>Technical Approach</h3>
  <ul>
    <li><strong>Base Model:</strong> Llama 3.1 70B</li>
    <li><strong>Training Framework:</strong> Unsloth (switched from FSDP due to memory constraints)</li>
    <li><strong>Training Data:</strong> Blog posts from nik.art (beginning through end of 2024)</li>
    <li><strong>Validation Data:</strong> Blog posts from 2025 onward</li>
    <li><strong>Checkpoints Evaluated:</strong> Epoch 800 and 1628</li>
  </ul>
  
  <h3>Evaluation Process</h3>
  <p>Each blog post was generated by both model checkpoints. Claude evaluated the quality of each generation and selected the better version when both were acceptable. The comparisons below show the original posts alongside the AI-generated versions that passed quality evaluation.</p>
  
  <p><strong>Repository:</strong> <a href="https://github.com/duc-ph/nik-llama" target="_blank">https://github.com/duc-ph/nik-llama</a></p>
</div>

"""

    sections = [head]

    for title, data in results.items():
        orig = html.escape(add_blank_lines(data["original"]))
        gen  = html.escape(add_blank_lines(data["generated"]))

        pairs = [("Original", orig), ("AI-Generated", gen)]
        random.shuffle(pairs)

        cols_html = "\n".join(
            f"""    <div class="article">
      <pre>{content}</pre>
      <span class="label hidden">{label}</span>
    </div>""" for label, content in pairs
        )

        sections.append(f"""
<div class="post">
  <h2>{html.escape(title)}</h2>

  <div class="comparison">
{cols_html}
  </div>

  <button class="toggle-btn" onclick="reveal(this)">Show source</button>
</div>
""")

    sections.append("</body>\n</html>")
    out_path.write_text("\n".join(sections), encoding="utf-8")
    print(f"✅  Comparison HTML written to: {out_path.resolve()}")

# Example usage:
# from your_module_with_results import final_result
make_html(final_result)